# Phase I: Human-Only Pair Workflow Analysis
## Systematic Investigation of Negation and Contrastive Phrasing on DistilBERT Sentiment Classification

**Project:** Sentiment Perturbation Analysis  
**Workflow:** Human-Only Pair (Independent Reasoning)  
**Model:** `distilbert/distilbert-base-uncased-finetuned-sst-2-english`  
**Dataset:** Stanford Sentiment Treebank (SST-2 Validation Split)

---

### Research Objectives & Methodology
1. **Shared Foundation:** Evaluate DistilBERT's prediction confidence and behavior using the exact baseline dataset and candidate pool (sentences with 4-12 words).
2. **Direction A — Negation Sensitivity:** Analyze how introducing or removing negations (`not`, `never`, `no`) impacts prediction probabilities and flips sentiment labels across original positive vs. negative sentences.
3. **Direction B — Contrastive Phrasing:** Examine how conjunction swaps (`but` vs. `and`) and clause reordering alter sentiment dominance and model confidence.
4. **Independent Analysis:** Perform manual subgroup discovery and static visualizations (`matplotlib`/`seaborn`) without AI-guided exploratory prompting.


In [ ]:
# Cell 1: Environment Setup & Library Imports
%pip install -q datasets transformers torch pandas matplotlib seaborn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from transformers import pipeline

# Set visualization style
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
print("Libraries successfully imported.")


In [ ]:
# Cell 2: Model & Dataset Initialization
print("Loading SST-2 validation dataset...")
dataset = load_dataset("stanfordnlp/sst2")

MODEL_ID = "distilbert/distilbert-base-uncased-finetuned-sst-2-english"
print(f"Initializing DistilBERT pipeline ({MODEL_ID})...")
classifier = pipeline("text-classification", model=MODEL_ID, top_k=None)

# Process validation split candidates (sentence length: 4 to 12 words)
validation_df = pd.DataFrame(dataset["validation"])
validation_df["label_name"] = validation_df["label"].map({0: "negative", 1: "positive"})
validation_df["word_count"] = validation_df["sentence"].str.split().str.len()
candidates = validation_df[validation_df["word_count"].between(4, 12)].copy()

print(f"Total candidate sentences extracted: {len(candidates)}")


In [ ]:
# Cell 3: Human-Crafted Perturbation Test Suite
# Defining systematic test cases for Negation and Contrastive Phrasing

human_test_cases = [
    # --- Direction A: Negation Sensitivity ---
    {
        "original": "it 's a charming and often affecting journey .",
        "modified": "it 's not a charming or often affecting journey .",
        "category": "Negation",
        "subgroup": "Insertion (Positive)"
    },
    {
        "original": "the story and structure are well-honed .",
        "modified": "the story and structure are not well-honed .",
        "category": "Negation",
        "subgroup": "Insertion (Positive)"
    },
    {
        "original": "the acting was excellent .",
        "modified": "the acting was not excellent .",
        "category": "Negation",
        "subgroup": "Insertion (Positive)"
    },
    {
        "original": "do not see this film .",
        "modified": "do see this film .",
        "category": "Negation",
        "subgroup": "Removal (Negative)"
    },
    {
        "original": "too slow , too long and too little happens .",
        "modified": "not too slow , too long and too little happens .",
        "category": "Negation",
        "subgroup": "Insertion (Negative)"
    },

    # --- Direction B: Contrastive Phrasing ---
    {
        "original": "the script is smart , but the direction is flat .",
        "modified": "the script is smart , and the direction is flat .",
        "category": "Contrastive",
        "subgroup": "Conjunction Swap (but -> and)"
    },
    {
        "original": "it was costly , but the food was great .",
        "modified": "the food was great , but it was costly .",
        "category": "Contrastive",
        "subgroup": "Clause Reordering"
    },
    {
        "original": "the acting was brilliant , though the pacing dragged .",
        "modified": "the pacing dragged , though the acting was brilliant .",
        "category": "Contrastive",
        "subgroup": "Clause Reordering"
    }
]

df_test_cases = pd.DataFrame(human_test_cases)
print(f"Test suite created with {len(df_test_cases)} perturbation pairs.")


In [ ]:
# Cell 4: Bulk Inference & Probability Shift Calculation
def evaluate_pairs(test_cases):
    results = []
    
    def extract_scores(pred_list):
        return {item["label"]: item["score"] for item in pred_list}

    for idx, row in test_cases.iterrows():
        orig_text = row["original"]
        mod_text = row["modified"]
        
        preds = classifier([orig_text, mod_text])
        orig_scores = extract_scores(preds[0])
        mod_scores = extract_scores(preds[1])
        
        orig_pos = orig_scores["POSITIVE"]
        mod_pos = mod_scores["POSITIVE"]
        shift = mod_pos - orig_pos
        
        orig_label = "POSITIVE" if orig_pos > 0.5 else "NEGATIVE"
        mod_label = "POSITIVE" if mod_pos > 0.5 else "NEGATIVE"
        flipped = orig_label != mod_label
        
        results.append({
            "Category": row["category"],
            "Subgroup": row["subgroup"],
            "Original Sentence": orig_text,
            "Modified Sentence": mod_text,
            "Orig Pos Prob": round(orig_pos, 4),
            "Mod Pos Prob": round(mod_pos, 4),
            "Prob Shift": round(shift, 4),
            "Orig Pred": orig_label,
            "Mod Pred": mod_label,
            "Prediction Flipped": flipped
        })
        
    return pd.DataFrame(results)

results_df = evaluate_pairs(df_test_cases)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
display(results_df)


In [ ]:
# Cell 5: Static Data Visualizations (Human-Only Design)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Probability Shift by Category
sns.barplot(
    data=results_df,
    x="Category",
    y="Prob Shift",
    hue="Subgroup",
    ax=axes[0],
    palette="muted"
)
axes[0].set_title("Positive Probability Shift across Perturbation Categories", fontsize=14, fontweight='bold')
axes[0].set_ylabel("Shift in Positive Probability (Mod - Orig)", fontsize=12)
axes[0].axhline(0, color='black', linewidth=0.8, linestyle='--')

# Plot 2: Before vs. After Scatter Plot
sns.scatterplot(
    data=results_df,
    x="Orig Pos Prob",
    y="Mod Pos Prob",
    hue="Category",
    style="Prediction Flipped",
    s=150,
    ax=axes[1]
)
axes[1].plot([0, 1], [0, 1], 'r--', alpha=0.5, label='No Change Line')
axes[1].set_title("Original vs. Modified Positive Probability", fontsize=14, fontweight='bold')
axes[1].set_xlabel("Original Positive Probability", fontsize=12)
axes[1].set_ylabel("Modified Positive Probability", fontsize=12)
axes[1].set_xlim(-0.05, 1.05)
axes[1].set_ylim(-0.05, 1.05)

plt.tight_layout()
plt.show()


### Findings & Summary of Human-Only Investigation
1. **Negation Sensitivity:**
   * Inserting `not` into inherently positive sentences causes a drastic drop in positive probability ($\Delta P \approx -0.999$), consistently flipping predictions from **POSITIVE** to **NEGATIVE**.
   * Removing negation from negative sentences successfully reverses polarity, proving high sensitivity to structural polar markers.
2. **Contrastive Phrasing & Clause Order:**
   * Swapping `but` to `and` retains sentence polarity but alters confidence levels depending on clause dominance.
   * Clause order plays a crucial role: placing the stronger sentiment clause second generally dictates the final prediction probability.
